- 실습 주제  
이전 노트북에서 저장한 Books to Scrape의 원본 HTML 파일을 불러와 BeautifulSoup으로 도서 정보 추출

- https://books.toscrape.com

- 작업 순서
원본 HTML 수집 완료
   - **원본 HTML 불러오기**
   - **BeautifulSoup 객체 생성**
   - **도서 한 건 추출**
   - **파싱 함수 작성**
   - **첫 페이지 전체 도서 추출**
   - **DataFrame 생성 및 기본 검증**

In [1]:
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from bs4.element import Tag

In [2]:
TARGET_URL = 'https://books.toscrape.com'
PROJECT_DIR = Path('D:/AI/data_analytics/crawling/01-data-collection-pipeline')
RAW_HTML_DIR = PROJECT_DIR / 'data' / 'raw' / 'html'

print(f'프로젝트 기준 경로 : {PROJECT_DIR}')
print(f'원본 HTML 저장 경로 : {RAW_HTML_DIR}')

프로젝트 기준 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline
원본 HTML 저장 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html


# 최신 원본 HTML 파일 찾기

```text
books_home_YYYYMMDD_HHMMSS.html
```

동일한 패턴의 파일이 여러 개 있을 수 있으므로 파일명 기준으로 
정렬한 뒤 가장 마지막 파일을 최신 원본 파일로 선택

In [3]:
pattern = 'books_home_*.html'

for r in RAW_HTML_DIR.glob(pattern):
    print(r)

D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\books_home_20260729_165900.html
D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\books_home_20260730_100728.html
D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\books_home_20260730_142258.html
D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\books_home_20260730_153908.html


In [4]:
## 정렬 : 오름차순
sorted(RAW_HTML_DIR.glob(pattern))

[WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260729_165900.html'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260730_100728.html'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260730_142258.html'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260730_153908.html')]

In [5]:
## 최근 파일
sorted(RAW_HTML_DIR.glob(pattern))[-1]

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260730_153908.html')

In [6]:
def find_latest_raw_html(
    directory: Path,
    pattern: str = 'books_home_*.html',
) -> Path:
    """
    지정한 폴더에서 파일명 패턴과 일치하는 최신 HTML 파일을 반환한다.

    Args:
        directory:
            원본 HTML 파일이 저장된 폴더

        pattern:
            검색할 파일명 패턴

    Returns:
        파일명 기준으로 마지막에 있는 HTML 파일 경로

    Raises:
        FileNotFoundError:
            폴더가 없거나 패턴에 맞는 HTML 파일이 없는 경우    
    """
    if not directory.exists():
        raise FileNotFoundError(f'원본 HTML 폴더가 없습니다. {directory}')

    html_files = sorted(directory.glob(pattern))

    if not html_files:
        raise FileNotFoundError('파싱할 원본 HTML 파일이 없습니다.')

    return html_files[-1]
    

In [7]:
latest_raw_file = find_latest_raw_html(RAW_HTML_DIR)
print(f'선택한 원본 HTML : {latest_raw_file}')

선택한 원본 HTML : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\raw\html\books_home_20260730_153908.html


# 원본 HTML 불러오기

원본 파일은 `response.content`를 `write_bytes()`로 저장했으므로 `read_bytes()`로 읽은 뒤 BeautifulSoup에 전달한다.

바이트 그대로 전달하면, BeautifulSoup이 문서 인코딩 정보를 참고하여 HTML을 분석할 수 있다.

In [8]:
def load_raw_html(file_path: Path) -> bytes:
    """
    원본 HTML 파일을 바이트 데이터 읽어 반환한다.

    Args:
        file_path:
            읽을 HTML 파일 경로

    Returns:
        HTML 원본 바이트 데이터

    Raises:
        FileNotFoundError:
            지정한 파일이 존재하지 않는 경우    
    """
    if not file_path.is_file():
        raise FileNotFoundError(f'HTML 파일이 없습니다. {file_path}')

    return file_path.read_bytes()

In [9]:
html_content = load_raw_html(latest_raw_file)
print(f'원본 HTML 크기 : {len(html_content):,} bytes')
print(html_content[:100])

원본 HTML 크기 : 51,294 bytes
b'<!DOCTYPE html>\n<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![end'


# BeautifulSoup 객체 생성

`BeautifulSoup()`은 원본 HTML을 탐색하고 필요한 요소를 선택할 수 있는 객체 구조로 변환한다.

```python
BeautifulSoup(HTML 데이터, 파서)
```

In [10]:
soup = BeautifulSoup(html_content)

print(type(soup))
print(soup.title)

<class 'bs4.BeautifulSoup'>
<title>
    All products | Books to Scrape - Sandbox
</title>


# 도서 상품 요소 전체 찾기

Books to Scrape의 목록 페이지에서 도서 구조는 아래와 같다.

```html
<article class="product_pod">
    ...
</article>
```

CSS 선택자 `article.product_pod`를 사용하여 도서 상품 요소를 모두 추출

In [11]:
products = soup.select('article.product_pod')
print(f'찾은 도서 수 : {len(products)}')

찾은 도서 수 : 20


# 도서 한 건 파싱
- 도서명
- 가격 원본 문자열
- 재고 상태 원본 문자열
- 평점 단어와 숫자 평점
- 상세 페이지 상대경로
- 상세 페이지 절대 URL

In [12]:
first_product = products[0]
print(type(first_product))
print()
print(first_product.prettify()[:100])

<class 'bs4.element.Tag'>

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-att


## 도서명

In [13]:
title_tag = first_product.select_one('h3 a')

if title_tag is None:
    raise ValueError('도서명 태그를 찾지 못했습니다.')

title = title_tag.get('title')

print(f'화면 표시 텍스트 : {title_tag.get_text()}')
print(f'title 속성값 : {title}')

화면 표시 텍스트 : A Light in the ...
title 속성값 : A Light in the Attic


## 가격

이번 단계에서는 통화 기호를 제거하지 않고 웹페이지의 원본 값을 유지

```text
£51.77
```

숫자형 변환은 다음 전처리 작업에서 수행

In [14]:
price_tag = first_product.select_one('.price_color')

if price_tag is None:
    raise ValueError('가격 태그를 찾지 못했습니다.')

price_text = price_tag.get_text(strip=True)    
print(f'가격 원본 문자열 : {price_text}')
print(f'자료형 : {type(price_text)}')

가격 원본 문자열 : £51.77
자료형 : <class 'str'>


## 재고 상태

재고 여부(`True`, `False`)로 변환하는 작업은 전처리 단계에서 수행

In [15]:
availability_tag = first_product.select_one('.availability')

if availability_tag is None:
    raise ValueError('재고 상태 태그를 찾지 못했습니다.')

availability_text = availability_tag.get_text(strip=True)

print(f'재고 상태 원본 문자열 : {availability_text}')
print(type(availability_text))

재고 상태 원본 문자열 : In stock
<class 'str'>


## 평점

평점은 태그의 텍스트가 아니라 클래스명에 있음

```html
<p class="star-rating Three">
    ...    
</p>
```
클래스 목록에서 `One`부터 `Five`까지의 값을 찾아 숫자로 변환

In [16]:
rating_tag = first_product.select_one('.star-rating')

if rating_tag is None:
    raise ValueError('평점 태그를 찾지 못했습니다.')

# print(rating_tag.get('href', '없음'))
rating_classes = rating_tag.get('class', []) ## class 속성이 없다면, [] 리턴
rating_classes

['star-rating', 'Three']

In [17]:
rating_text = rating_classes[-1]
rating_text

'Three'

In [18]:
RATING_MAP = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

RATING_MAP[rating_text]
rating = RATING_MAP.get(rating_text)
rating

3

In [19]:
print(f'평점 클래스 목록 : {rating_classes}')
print(f'평점 단어 : {rating_text}')
print(f'숫자 평점 : {rating}')

평점 클래스 목록 : ['star-rating', 'Three']
평점 단어 : Three
숫자 평점 : 3


## 상세 페이지 URL

`href` 속성의 값은 상대 URL

```text
catalogue/a-light-in-the-attic_1000/index.html
```

`urljoin()`을 사용하면 기준 URL과 상대 URL을 결합하여 절대 URL로 변환

In [20]:
detail_path = title_tag.get('href')

if detail_path is None:
    raise ValueError('상세 페이지 상대 URL을 찾지 못했습니다.')

detail_url = urljoin(TARGET_URL, detail_path)
detail_url

print(f'상대 URL : {detail_path}')
print(f'절대 URL : {detail_url}')

상대 URL : catalogue/a-light-in-the-attic_1000/index.html
절대 URL : https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


## 도서 한 건을 딕셔너리에 저장

HTML 태그 객체 자체를 저장하지 않고, 이후 정제와 저장에 사용할 수 있도록 필요한 값만 딕셔너리로 구성

In [21]:
first_book = {
    'title': title,
    'price_text': price_text,
    'availability_text': availability_text,
    'rating_text': rating_text,
    'rating': rating,
    'detail_path': detail_path,
    'detail_url': detail_url,
}

first_book

{'title': 'A Light in the Attic',
 'price_text': '£51.77',
 'availability_text': 'In stock',
 'rating_text': 'Three',
 'rating': 3,
 'detail_path': 'catalogue/a-light-in-the-attic_1000/index.html',
 'detail_url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}

# 파싱 함수 정의

지금까지 작성한 도서 한 건 추출 코드를 다음 함수로 분리

- `get_required_tag()`: 필수 HTML 태그 선택
- `parse_rating()` : 평점 클래스 추출 및 숫자 변환
- `parse_book_item()` : 도서 한 건 파싱

## 필수 태그 선택 함수 정의 : get_required_tag()

In [22]:
def get_required_tag(
    parent: Tag,
    selector: str,
    field_name: str
) -> Tag:
    """
    부모 태그에서 필수하위 태그를 찾아 반환한다.

    Args:
        parent:
            검색 기준이 되는 부모 HTML 태그

        selector:
            찾을 CSS 선택자

        field_name
            오류 메시지에 표시할 필드명

    Returns:
        선택자와 일치하는 첫 번째 HTML 태그

    Raises:
       ValueError:
           필수 태그를 찾지 못한 경우    
    """
    tag = parent.select_one(selector)

    if tag is None:
        raise ValueError(
            f'{field_name} 태그를 찾지 못했습니다. '
            f'선택자 : {selector}'
        )

    return tag    

## 평점 파싱 함수 정의 : parse_rating()

In [23]:
def parse_rating(rating_tag: Tag) -> tuple[str, int]:
    """
    평점 태그의 클래스에서 평점 단어와 숫자 평점을 추출한다.

    Args:
        rating_tag:
            start-rating 클래스가 있는 HTML 태그

    Returns:
        평점 단어와 숫자 평점의 튜플

    Raises:
        ValueError:
            One부터 Five까지의 평점 클래스를 찾지 못한 경우    
    """

    rating_classes = rating_tag.get('class', []) ## class 속성이 없다면, [] 리턴
    rating_text = rating_classes[-1]

    if rating_text is None:
        raise ValueError(f'유효한 평점 클래스를 찾지 못했습니다. : {rating_classes}')

    return (rating_text, RATING_MAP[rating_text])

    

## 도서 한 건 파싱 함수 정의 : parse_book_item()

In [24]:
def parse_book_item(
    product: Tag,
    base_url: str,
) -> dict[str, str | int]:
    """
    도서 상품 HTML 요소 한 개에서 도서 페이지 정보를 추출한다.

    Args:
        product:
            article.product_pod 도서 상품 요소

        base_url:
            상대 URL을 절대 URL로 변환할 기준 URL

    Returns:
        도서 한 건의 파싱 결과 딕셔너리
    
    Raises:
        ValueError:
            필수 태그나 필수 속성을 찾지 못한 경우    
    """

    title_tag = get_required_tag(product, 'h3 a', '도서명')
    price_tag = get_required_tag(product, '.price_color', '가격')
    availability_tag = get_required_tag(product, '.availability', '재고 상태')
    rating_tag = get_required_tag(product, '.star-rating', '평점')

    title = title_tag.get('title')
    detail_path = title_tag.get('href')

    if not title:
        raise ValueError('도서명의 title 속성이 없습니다.')

    if not detail_path:
        raise ValueError('상세 페이지 href 속성이 없습니다.')

    rating_text, rating = parse_rating(rating_tag)

    book_info = {
        'title': title,
        'price_text': price_tag.get_text(strip=True),
        'availability_text': availability_tag.get_text(strip=True),
        'rating_text': rating_text,
        'rating': rating,
        'detail_path': detail_path,
        'detail_url': urljoin(base_url, detail_path)
    }

    return book_info   

## 파싱 함수 동작 확인
- 파싱 함수로 첫 번째 도서의 정보 추출 결과와 앞에서 개별로 추출한 결과와 일치하는 확인

In [26]:
parsed_first_book = parse_book_item(first_product, TARGET_URL)

parsed_first_book

{'title': 'A Light in the Attic',
 'price_text': '£51.77',
 'availability_text': 'In stock',
 'rating_text': 'Three',
 'rating': 3,
 'detail_path': 'catalogue/a-light-in-the-attic_1000/index.html',
 'detail_url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}

In [27]:
assert parsed_first_book == first_book

print('함수의 결과와 개별 추출 결과가 일치합니다.')

함수의 결과와 개별 추출 결과가 일치합니다.


# 첫 페이지 전체 도서 파싱

In [33]:
books = []

for product in products:
    book = parse_book_item(product, TARGET_URL)
    books.append(book)

print(f'파싱한 도서 수 : {len(books)}')

파싱한 도서 수 : 20


# DataFrame 생성

딕셔너리 리스트를 Pandas DataFrame으로 변환

현재 DataFrame은 아직 정제 전 상태이므로 다음 원본 컬럼을 유지한다.
- `price_text`
- `availability_text`
- `rating_text`
- `detail_path`

In [35]:
books_df = pd.DataFrame(books)
books_df.shape

(20, 7)

In [36]:
books_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   title              20 non-null     str  
 1   price_text         20 non-null     str  
 2   availability_text  20 non-null     str  
 3   rating_text        20 non-null     str  
 4   rating             20 non-null     int64
 5   detail_path        20 non-null     str  
 6   detail_url         20 non-null     str  
dtypes: int64(1), str(6)
memory usage: 1.2 KB


# 파싱 결과 기본 검증

이번 단계의 검증은 HTML에서 필요한 데이터가 빠짐없이 추출되었는지 확인하는 데 목적이 있다.

본격적인 자료형 변환 비즈니스 규칙 검증은 다음 전처리 단계에서 수행한다.


## 필수 컬럼 검증

In [ ]:
REQUIRED_COLUMNS = [
    'title', 
    'price_text', 
    'availability_text', 
    'rating_text', 
    'rating',
    'detail_path',
    'detail_url'
]

In [ ]:
missing_columns = [
    column 
    for column in REQUIRED_COLUMNS
    if column not in books_df.columns
]

if missing_columns:
    raise ValueError(f'필수 컬럼이 누락되었다. {missing_columns}')

print('필수 컬럼 검증 완료')

필수 컬럼 검증 완료


## 필수 데이터 결측값 검증

In [ ]:
null_couts = books_df[REQUIRED_COLUMNS].isna().sum()
null_couts
null_couts.sum()

if null_couts.sum() > 0:
    raise ValueError(f'필수 데이터에 결측값이 있습니다. {null_couts[null_couts > 0]}')

print('필수 데이터 결측값 검증 완료')

필수 데이터 결측값 검증 완료


## 평점 범위 검증

In [57]:
~books_df['rating'].between(1, 5)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: rating, dtype: bool

In [61]:
invaild_ratings = books_df[~books_df['rating'].between(1, 5)]

invaild_ratings
invaild_ratings.empty

if not invaild_ratings.empty:
    raise ValueError('1부터 5 범위를 벗어난 평점이 있습니다.')
    
print('평점 범위 검증 완료')

평점 범위 검증 완료


## 중복 상세 URL 수 확인

In [72]:
duplicate_detail_urls = books_df.detail_url.duplicated(keep=False)
duplicate_detail_urls.sum()

print('중복 상세 URL 수 :', int(duplicate_detail_urls.sum()))

중복 상세 URL 수 : 0


# DataFrame을 CSV 파일로 저장

In [73]:
PROJECT_DIR

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline')

In [76]:
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
INTERIM_DIR

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

from datetime import datetime

save_at = datetime.now().strftime('%Y%m%d_%H%M%S')
parsed_file = INTERIM_DIR / f'books_page_001_parsed_{save_at}.csv'
books_df.to_csv(parsed_file, index=False, encoding='utf-8-sig')
print(f'첫 페이지 파싱 데이터 저장 완료 : {parsed_file}')

첫 페이지 파싱 데이터 저장 완료 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\interim\books_page_001_parsed_20260731_143406.csv


# 파싱 결과 요약

In [77]:
print('=' * 60)
print('정적 웹페이지 파싱 결과')
print('=' * 60)

print(f'원본 HTML 파일 : {latest_raw_file.name}')
print(f'HTML 도서 수 : {len(products)}')
print(f'파싱한 도서 수 : {len(books_df)}')
print(f'도서명 결측 수 : {books_df.title.isna().sum()}')
print(f'상세 URL 중복 수 : {duplicate_detail_urls.sum()}')
print(f'평점 최솟값 : {books_df.rating.min()}')
print(f'평점 최댓값 : {books_df.rating.max()}')

정적 웹페이지 파싱 결과
원본 HTML 파일 : books_home_20260730_153908.html
HTML 도서 수 : 20
파싱한 도서 수 : 20
도서명 결측 수 : 0
상세 URL 중복 수 : 0
평점 최솟값 : 1
평점 최댓값 : 5


# 저장 후 재읽기 검증

In [80]:
saved_books_df = pd.read_csv(parsed_file, dtype=str)

if len(saved_books_df) != len(books_df):
    raise ValueError('csv 저장 전후의 행 수가 다릅니다.')

print(f'csv 저장, 재읽기 검증 완료 : {len(saved_books_df)}')

csv 저장, 재읽기 검증 완료 : 20


# 이번 노트북에서 완료한 작업

```text
저장된 원본 HTML 불러오기
-> BeautifulSoup 객체 생성
-> 도서 요소 찾기
-> 첫 번째 도서 정보 추출
-> 도서 한 건 딕셔너리 생성
-> 파싱 함수 작성
-> 첫 페이지 전체 도서 파싱
-> DataFrame 생성
-> 필수 컬럼, 결측값, 평점, 중복 URL 검증
-> 첫 페이지 파싱 데이터 csv 저장
-> 저장 결과 재읽기 검증
```